In [10]:
!pip install transformers datasets evaluate jiwer accelerate -q
!pip install torch torchaudio -q
!pip install gradio -q
#run

In [11]:
#run
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os


ZIP_PATH = '/content/drive/MyDrive/finetuning/1774126233883-cv-corpus-25.0-2026-03-09-hi.tar.gz'
EXTRACT_TO = '/content/hindi_data/'

os.makedirs(EXTRACT_TO, exist_ok=True)


import tarfile
with tarfile.open(ZIP_PATH, 'r:gz') as t:
    t.extractall(EXTRACT_TO)

print("Extracted! Files:")
os.listdir(EXTRACT_TO)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_5480/3285997802.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(EXTRACT_TO)


Extracted! Files:


['cv-corpus-25.0-2026-03-09']

In [12]:
import os

# find train.tsv
for root, dirs, files in os.walk('/content/hindi_data'):
    for f in files:
        if f in ['train.tsv', 'test.tsv']:
            print(os.path.join(root, f))

/content/hindi_data/cv-corpus-25.0-2026-03-09/hi/train.tsv
/content/hindi_data/cv-corpus-25.0-2026-03-09/hi/test.tsv


In [13]:
import pandas as pd

# Update DATA_DIR
DATA_DIR = '/content/hindi_data/cv-corpus-25.0-2026-03-09/hi/'

train_df = pd.read_csv(DATA_DIR + 'train.tsv', sep='\t')
test_df  = pd.read_csv(DATA_DIR + 'test.tsv',  sep='\t')

print(f"Train samples: {len(train_df)}")
print(f"Test  samples: {len(test_df)}")
print(train_df[['path', 'sentence']].head(3))

Train samples: 4894
Test  samples: 3326
                           path                                           sentence
0  common_voice_hi_26008353.mp3                           हमने उसका जन्मदिन मनाया।
1  common_voice_hi_26010468.mp3  साउथ दिल्ली नगर निगम सख्त, शॉपिंग मॉल के बाहर ...
2  common_voice_hi_26010469.mp3         उत्तर कोरिया ने अमेरिका को दी हमले की धमकी


In [14]:
import torch, torchaudio
from datasets import Dataset, Audio

CLIPS_DIR = DATA_DIR + 'clips/'

def build_dataset(df, clips_dir, max_samples=2000):
    df = df.dropna(subset=['path', 'sentence']).head(max_samples)
    df['audio_path'] = df['path'].apply(lambda p: clips_dir + p)
    df = df[df['audio_path'].apply(lambda p: os.path.exists(p))]
    hf = Dataset.from_dict({
        'audio': df['audio_path'].tolist(),
        'sentence': df['sentence'].tolist()
    })
    hf = hf.cast_column('audio', Audio(sampling_rate=16000))
    return hf

print("Building train set (up to 2000 samples)...")
train_dataset = build_dataset(train_df, CLIPS_DIR, max_samples=2000)
print("Building test set (up to 300 samples)...")
test_dataset  = build_dataset(test_df,  CLIPS_DIR, max_samples=300)

print(f"Train: {len(train_dataset)}  Test: {len(test_dataset)}")

Building train set (up to 2000 samples)...
Building test set (up to 300 samples)...
Train: 2000  Test: 300


In [15]:
#model load
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = 'openai/whisper-small'

print("Loading processor...")
processor = WhisperProcessor.from_pretrained(MODEL_NAME)

print("Loading model...")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language='hindi', task='transcribe'
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f"Model loaded on: {device}")

Loading processor...
Loading model...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model loaded on: cuda


In [19]:
import torch

def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = processor(
        audio['array'],
        sampling_rate=audio['sampling_rate'],
        return_tensors='pt'
    ).input_features[0]
    batch['labels'] = processor.tokenizer(
        batch['sentence'],
        return_tensors='pt'
    ).input_ids[0]
    return batch

print("Preprocessing train set...")
train_processed = train_dataset.map(
    prepare_dataset,
    remove_columns=train_dataset.column_names,
    desc='Preprocessing train'
)

print("Preprocessing test set...")
test_processed = test_dataset.map(
    prepare_dataset,
    remove_columns=test_dataset.column_names,
    desc='Preprocessing test'
)

print("Done preprocessing!")

Preprocessing train set...


Preprocessing train:   0%|          | 0/2000 [00:00<?, ? examples/s]

Preprocessing test set...


Preprocessing test:   0%|          | 0/300 [00:00<?, ? examples/s]

Done preprocessing!


In [18]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors='pt'
        )
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors='pt'
        )
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data collator ready.")

Data collator ready.


In [16]:
import evaluate

wer_metric = evaluate.load('wer')
cer_metric = evaluate.load('cer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': round(wer, 2), 'cer': round(cer, 2)}

print("Ready metrics")

Ready metrics


In [20]:
from torch.utils.data import DataLoader

def quick_evaluate(model, dataset, processor, num_samples=100, batch_size=8):
    model.eval()
    all_preds, all_refs = [], []
    subset = dataset.select(range(min(num_samples, len(dataset))))
    loader = DataLoader(subset, batch_size=batch_size,
                        collate_fn=data_collator)
    for batch in loader:
        input_features = batch['input_features'].to(device)
        labels = batch['labels']
        labels[labels == -100] = processor.tokenizer.pad_token_id
        with torch.no_grad():
            pred_ids = model.generate(input_features)
        pred_str  = processor.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = processor.batch_decode(labels,   skip_special_tokens=True)
        all_preds.extend(pred_str)
        all_refs.extend(label_str)
    wer = 100 * wer_metric.compute(predictions=all_preds, references=all_refs)
    cer = 100 * cer_metric.compute(predictions=all_preds, references=all_refs)
    return round(wer, 2), round(cer, 2)

print("Evaluating BASE model (before fine-tuning)...")
base_wer, base_cer = quick_evaluate(model, test_processed, processor)
print(f"BASE MODEL  →  WER: {base_wer}%   CER: {base_cer}%")

Evaluating BASE model (before fine-tuning)...
BASE MODEL  →  WER: 101.18%   CER: 68.34%


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

OUTPUT_DIR = '/content/drive/MyDrive/whisper-hindi-finetuned'

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=100,
    num_train_epochs=3,
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=50,
    report_to=['none'],
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_processed,
    eval_dataset=test_processed,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print("Starting training...")
trainer.train()
print("Training complete!")

Starting training... (takes ~1-2 hours on T4 GPU)


Epoch,Training Loss,Validation Loss,Wer,Cer
1,1.216605,0.578488,54.590000,23.200000
2,0.422884,0.404992,47.110000,16.650000
3,0.229693,0.412090,47.110000,17.250000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Training complete!


In [9]:
import gradio as gr
import numpy as np
import torch

def transcribe_audio(audio):
    if audio is None:
        return "No audio received. Please record something."
    try:
        sr, data = audio
        if data.ndim > 1:
            data = data.mean(axis=1)
        if data.dtype == np.int16:
            data = data.astype(np.float32) / 32768.0
        elif data.dtype == np.int32:
            data = data.astype(np.float32) / 2147483648.0
        elif data.dtype != np.float32:
            data = data.astype(np.float32)
        if sr != 16000:
            import torchaudio
            data_tensor = torch.tensor(data).unsqueeze(0)
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            data = resampler(data_tensor).squeeze(0).numpy()
            sr = 16000

        input_features = processor(
            data, sampling_rate=sr, return_tensors="pt"
        ).input_features.to(device)

        forced_decoder_ids = processor.get_decoder_prompt_ids(
            language="hindi",
            task="transcribe"
        )

        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                forced_decoder_ids=forced_decoder_ids
            )

        result = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        return result if result.strip() else "Could not transcribe. Try speaking louder."

    except Exception as e:
        return f"Error: {str(e)}"

demo = gr.Interface(
    fn=transcribe_audio,
    inputs=gr.Audio(sources=["microphone", "upload"], label="Speak Hindi"),
    outputs=gr.Textbox(label="Transcription in hindi"),
    title="Hindi Speech-to-Text (Fine-tuned Whisper)",
    description="Speak Hindi into the mic — transcription will appear in Devanagari script (hindi)"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://108d034614f5de596a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
from torch.utils.data import DataLoader

def quick_evaluate(mdl, dataset, num_samples=100):
    mdl.eval()
    all_preds, all_refs = [], []
    subset = dataset.select(range(min(num_samples, len(dataset))))
    loader = DataLoader(subset, batch_size=8, collate_fn=data_collator)
    for batch in loader:
        input_features = batch['input_features'].to(device)
        labels = batch['labels'].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        with torch.no_grad():
            pred_ids = mdl.generate(input_features)
        all_preds.extend(processor.batch_decode(pred_ids, skip_special_tokens=True))
        all_refs.extend(processor.batch_decode(labels,    skip_special_tokens=True))
    wer = round(100 * wer_metric.compute(predictions=all_preds, references=all_refs), 2)
    cer = round(100 * cer_metric.compute(predictions=all_preds, references=all_refs), 2)
    return wer, cer

# Base model
print("Evaluating base model...")
from transformers import WhisperForConditionalGeneration
base_model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small')
base_model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language='hindi', task='transcribe')
base_model = base_model.to(device)
base_wer, base_cer = quick_evaluate(base_model, test_processed)

# Fine-tuned model
print("Evaluating fine-tuned model...")
ft_wer, ft_cer = quick_evaluate(model, test_processed)

# Print results
print("\n" + "="*50)
print("         RESULTS COMPARISON")
print("="*50)
print(f"{'Metric':<10} {'Base':>10} {'Fine-tuned':>12} {'Improvement':>13}")
print("-"*50)
print(f"{'WER':<10} {base_wer:>9}% {ft_wer:>11}% {round(base_wer-ft_wer,2):>+12}%")
print(f"{'CER':<10} {base_cer:>9}% {ft_cer:>11}% {round(base_cer-ft_cer,2):>+12}%")
print("="*50)


Evaluating base model...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Evaluating fine-tuned model...

         RESULTS COMPARISON
Metric           Base   Fine-tuned   Improvement
--------------------------------------------------
WER           101.18%       47.36%       +53.82%
CER            68.34%       17.17%       +51.17%


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/whisper-hindi-finetuned/final_model'
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print(f"Model saved to: {SAVE_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/whisper-hindi-finetuned/final_model


In [22]:
# run this cell  when reopening after training is done
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

SAVE_DIR = '/content/drive/MyDrive/whisper-hindi-finetuned/final_model'

processor = WhisperProcessor.from_pretrained(SAVE_DIR)
model = WhisperForConditionalGeneration.from_pretrained(SAVE_DIR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

print(f"Fine-tuned model loaded from Drive on: {device}")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Fine-tuned model loaded from Drive on: cuda
